## Imports

In [1]:
import pandas as pd
import os
import numpy as np
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve, confusion_matrix
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline


## Config

In [2]:
DATA_FILE  = "../../data/processed/cardio_onc_prostate_04cleaned.csv"
OUT_DIR  = "Results/GradientBoosting"
TARGET = "at_risk"
N_FOLDS = 5
SEED = 42

os.makedirs(OUT_DIR, exist_ok=True)

## Load Data

In [3]:
df = pd.read_csv(DATA_FILE)
print(f"Raw shape: {df.shape}")

Raw shape: (239, 53)


In [4]:
# 2. Drop rows where TARGET is missing (n=36)
# ─────────────────────────────────────────────────────────────────────────────
df = df[df[TARGET].notna()].copy()
print(f"Shape after dropping missing target: {df.shape}")
print(f"Class distribution — 0: {(df[TARGET] == 0).sum()}  1: {(df[TARGET] == 1).sum()}")

Shape after dropping missing target: (203, 53)
Class distribution — 0: 124  1: 79


## Define Feature Columns

Excluded:
- unique_patient_id — row identifier
- ethnicity, specific_nht_used, — raw strings (encoded versions kept)
- adt_agent, prescribing_provider
- nht_auth_date, nht_start_date, — raw datetimes (derived numerics kept)
- adt_start_date
- at_risk — target

In [5]:
continuous_features = [
    "bmi", "age", "sbp", "dbp", "ascvd_10yr",
    "days_auth_to_start", "days_adt_to_nht",
]

binary_features = [
    "hx_smoking", "family_hx_hd", "hx_htn",
    "bp_meds_prior", "bb_prior", "ace_arb_prior",
    "has_pcp", "hx_hld", "hx_high_tg",
    "statin_prior", "other_lipid_prior", "lipid_panel_checked",
    "hx_cad", "hx_mi_stent", "hx_chf", "hx_arrhythmia",
    "hx_carotid", "hx_pad", "hx_cva",
    "hx_dm2", "dm_noninsulin", "on_insulin",
    "a1c_checked", "glucose_over_200",
    "asa_use", "cards_prior", "cards_post", "cards_referral",
    "diet_counseling", "exercise_counseling",
    "echo_ordered", "ecg_done", "non_onc_provider",
]

encoded_features = [
    "ethnicity_enc", "specific_nht_used_enc",
    "adt_agent_enc", "prescribing_provider_enc",
]

all_features = continuous_features + binary_features + encoded_features

# Sanity check
missing_cols = [c for c in all_features if c not in df.columns]
if missing_cols:
    print(f"WARNING — columns not found in dataframe: {missing_cols}")
else:
    print(f"\nAll {len(all_features)} feature columns confirmed present in dataframe.")

print(f"  Continuous:    {len(continuous_features)}")
print(f"  Binary:        {len(binary_features)}")
print(f"  Encoded cats:  {len(encoded_features)}")


All 44 feature columns confirmed present in dataframe.
  Continuous:    7
  Binary:        33
  Encoded cats:  4


## Build X, y

In [6]:
X_df = df[all_features].copy().astype(float)
y = df[TARGET].astype(int)

miss = X_df.isnull().sum()
miss = miss[miss > 0].sort_values(ascending = False)
print(f"\nFeature-matrix missingness (non-zero columns only):")
print(miss.to_string())
print(f"\nComplete cases if listwise deletion: {X_df.dropna().shape[0]} / {X_df.shape[0]}")
print("→ Using imputation instead of listwise deletion.\n")

X = X_df


Feature-matrix missingness (non-zero columns only):
days_adt_to_nht       14
adt_agent_enc         11
ethnicity_enc          4
days_auth_to_start     1

Complete cases if listwise deletion: 185 / 203
→ Using imputation instead of listwise deletion.



## Model Definitions

In [7]:
cv_splitter = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)

models = {
    "xgboost": xgb.XGBClassifier(
        n_estimators=500,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
    )
}

In [8]:
summary_rows = []
fitted_models = {}

scores = []

scoring_metrics = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "accuracy": "accuracy",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
}

# -----------------------------
# CROSS VALIDATION LOOP
# -----------------------------
for name, model in models.items():

    print(f"\n{'='*60}")
    print(f"Model: {name}")
    print(f"{'='*60}")

    cv_results = cross_validate(
        model,
        X,
        y,
        cv=cv_splitter,
        scoring=scoring_metrics,
        return_train_score=True,
        n_jobs=-1
    )

    row = {"model": name}

    for metric in scoring_metrics:
        vals = cv_results[f"test_{metric}"]
        mean = vals.mean()
        std = vals.std()

        row[f"{metric}_mean"] = mean
        row[f"{metric}_std"] = std

        print(f"{metric:<20} {mean:.4f} ± {std:.4f}")

    summary_rows.append(row)

    # -----------------------------
    # FIT FINAL MODEL ON FULL DATA
    # -----------------------------
    model.fit(X, y)
    fitted_models[name] = model

# -----------------------------
# SUMMARY TABLE
# -----------------------------
summary_df = pd.DataFrame(summary_rows)
print("\nFinal CV Summary:")
print(summary_df)

summary_df.to_csv(os.path.join(OUT_DIR, "cv_summary.csv"), index=False)
print(f"\nSaved → {os.path.join(OUT_DIR, 'cv_summary.csv')}")


Model: xgboost
roc_auc              0.6433 ± 0.0360
average_precision    0.5579 ± 0.0567
accuracy             0.6500 ± 0.0728
f1                   0.5025 ± 0.1000
recall               0.4550 ± 0.0967
precision            0.5667 ± 0.1169

Final CV Summary:
     model  roc_auc_mean  roc_auc_std  average_precision_mean  \
0  xgboost      0.643254     0.036039                0.557854   

   average_precision_std  accuracy_mean  accuracy_std   f1_mean    f1_std  \
0               0.056698           0.65      0.072823  0.502452  0.100031   

   recall_mean  recall_std  precision_mean  precision_std  
0        0.455    0.096695        0.566667       0.116905  

Saved → Results/GradientBoosting\cv_summary.csv


## SHAP Values

In [9]:
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

model = fitted_models["xgboost"]
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X, show=False, max_display=5)
#shap.summary_plot(shap_values, X, plot_type="bar", show=False)

plt.title("Top 5 Predictors by SHAP Value")

plt.savefig(
    os.path.join(OUT_DIR, "shap_summary.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print(f"\nSaved → {os.path.join(OUT_DIR, 'shap_summary.png')}")


Saved → Results/GradientBoosting\shap_summary.png


In [10]:
shap_df = pd.DataFrame(shap_values, columns=X.columns)

shap_coef_table = pd.DataFrame({
    "feature": X.columns,
    "mean_abs_shap": np.abs(shap_df).mean(),
    "mean_shap": shap_df.mean()
}).sort_values("mean_abs_shap", ascending=False)

shap_coef_table.to_csv(
    os.path.join(OUT_DIR, "shap_feature_table.csv"),
    index=False
)

print(f"\nSaved → {os.path.join(OUT_DIR, 'shap_feature_table.csv')}")

shap_coef_table



Saved → Results/GradientBoosting\shap_feature_table.csv


,feature,mean_abs_shap,mean_shap
dm_noninsulin,dm_noninsulin,0.940560,-0.132485
days_adt_to_nht,days_adt_to_nht,0.684085,-0.146335
bmi,bmi,0.558253,-0.098517
bp_meds_prior,bp_meds_prior,0.427783,-0.054633
lipid_panel_checked,lipid_panel_checked,0.425126,0.003267
age,age,0.328410,-0.064196
days_auth_to_start,days_auth_to_start,0.324553,-0.042648
adt_agent_enc,adt_agent_enc,0.324541,-0.040771
hx_smoking,hx_smoking,0.313031,-0.023889
ethnicity_enc,ethnicity_enc,0.236788,-0.005663


## Confusion Matrices

In [11]:
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

fig, ax = plt.subplots(1, 1, figsize=(5, 3))

model = models["xgboost"]

y_pred = cross_val_predict(
    model,
    X,
    y,
    cv=cv_splitter,
    method="predict_proba"
)[:, 1]

# apply threshold
y_pred_label = (y_pred > 0.5).astype(int)

# Confusion matrix
cm = confusion_matrix(y, y_pred_label)

table = ax.table(
    cellText=cm,
    rowLabels=["True 0", "True 1"],
    colLabels=["Pred 0", "Pred 1"],
    loc="center",
    cellLoc="center"
)

table.scale(1.5, 1.5)
ax.set_title("XGBoost", fontsize=11)
ax.axis("off")

# Print classification report
print("\nXGBoost — Classification Report (cross-validated predictions):")
print(classification_report(y, y_pred_label))

plt.suptitle("Confusion Matrix (cross-validated, threshold=0.5)")
plt.tight_layout()

plt.savefig(
    os.path.join(OUT_DIR, "confusion_matrix.png"),
    dpi=300,
    bbox_inches="tight"
)
plt.close()

print(f"\nSaved → {os.path.join(OUT_DIR, 'confusion_matrix.png')}")


XGBoost — Classification Report (cross-validated predictions):
              precision    recall  f1-score   support

           0       0.68      0.75      0.72       124
           1       0.54      0.46      0.49        79

    accuracy                           0.64       203
   macro avg       0.61      0.60      0.60       203
weighted avg       0.63      0.64      0.63       203


Saved → Results/GradientBoosting\confusion_matrix.png
